In [ ]:
from forestkernel import ForestKernel
# from rfgap import rfgap_new import RFGAP


from sklearn.model_selection import train_test_split
from dataset import dataprep


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Read in the data and normalize

In [ ]:
# TODO: Test with NumPy and Pandas, ALL METHODS and perhaps combos of each type. Problem with categorical data in Pandas?
# We should probably modify RFGAP to handle categorical data in the form of strings.

In [ ]:
seed = 42
test_size = 0.2
val_size = 0.2
kernel_method = 'gap'
model_type = 'rf'
force_symmetric = False
force_nonzero_diag = True
normalize_diagonal = False
oob_score = True
max_samples = None
verbose = 1
n_jobs = -1

data = pd.read_csv('./data/iris.csv')
x, y = dataprep(data)

n_samples = x.shape[0]
n_features = x.shape[1]

# 1) Split indices first
all_idx = np.arange(n_samples)
train_idx, test_idx = train_test_split(
    all_idx,
    test_size=test_size,
    random_state=seed,
    stratify=y
)

# 2) Build train/test sets from indices
x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 3) Mask random train indices
perc = 0.30
y_train_masked = np.asarray(y_train).copy()

n_mask = int(perc * y_train.shape[0])
mask_idx_in_train = np.arange(y_train.shape[0] - n_mask, y_train.shape[0])

# Optional: original dataset indices that were masked
masked_original_idx = train_idx[mask_idx_in_train]

## Train the RF Model

In [ ]:
rf = ForestKernel(y = y_train,
                  kernel_method = kernel_method,
                  matrix_type = 'sparse',
                  oob_score = True,
                  max_samples = max_samples,
                  force_nonzero_diag = force_nonzero_diag,
                  random_state = seed,
                  model_type=model_type,
                  n_jobs = n_jobs,
                  force_symmetric = force_symmetric,
                #   allow_semi_supervised=allow_semi_supervised,
                  normalize_diagonal=normalize_diagonal,
                  verbose=verbose)

In [ ]:
rf.fit(x_train, y_train, mask_idx_in_train)
y_train_masked

## Generate the Kernel Matrix

In [ ]:
# compute/get proximity matrix and visualize with seaborn heatmap
prox = rf.get_kernel()
# prox = rf.kernel_extend(x_train)

# Show diagonal values
diag = prox.diagonal()
print(diag)

# Visualize it

In [ ]:
prox_mat = prox.toarray()

# mask diagonal to emphasize off-diagonal proximities
# mask = np.eye(prox_mat.shape[0], dtype=bool)
mask=None

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(prox_mat, ax=ax, cmap='rocket_r', vmin=0, vmax=prox_mat.max(),
            mask=mask,
            xticklabels=False, yticklabels=False, square=True, cbar_kws={'label': 'Proximity'})
ax.set_title(f'{kernel_method.upper()} Proximity Matrix')
plt.show()

# Check row sums
row_sums = prox_mat.sum(axis=1)
print("Row sums :")
print(row_sums)

# Check symmetry
is_symmetric = np.allclose(prox_mat, prox_mat.T, atol=1e-6)
print(f"Is the proximity matrix symmetric? {is_symmetric}")

## Check extended kernel computation

In [ ]:
# selected_train_indices = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
test_prox = rf.kernel_extend(x_test).toarray()
selected_train_indices = np.arange(test_prox.shape[1])
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(test_prox, ax=ax, cmap='rocket_r', vmin=0, vmax=test_prox.max(),
            xticklabels=[f'train_{i}' for i in selected_train_indices],
            yticklabels=False, cbar_kws={'label': 'Proximity'})
ax.set_xticklabels(ax.get_xticklabels(), fontsize=5) # 'ha' aligns text nicely with rotation
ax.set_xlabel('Selected training indices')
ax.set_ylabel('Test samples')
ax.set_title(f'{kernel_method.upper()} Test-to-Train Proximity (shape={test_prox.shape})')
plt.tight_layout()
plt.show()